In [ ]:
import os
from git import Repo
import re
import json
from tqdm import tqdm

# === CONFIG ===
LOCAL_REPO_DIR = "./nasa_repos"
SECURITY_KEYWORDS = [
    "cve", "cwe", "security", "vulnerability", "vulnerable","buffer overflow",
    "fix", "patch"
]

# === HELPER FUNCTION ===
def is_security_commit(message):
    return any(kw.lower() in message.lower() for kw in SECURITY_KEYWORDS)

# === PROCESS COMMITS IN EACH REPO ===
all_security_commits = []

for repo_name in tqdm(os.listdir(LOCAL_REPO_DIR), desc="Scanning Repos"):
    repo_path = os.path.join(LOCAL_REPO_DIR, repo_name)
    if not os.path.isdir(repo_path):
        continue
    try:
        repo = Repo(repo_path)
        for commit in repo.iter_commits():
            msg = commit.message
            if is_security_commit(msg):
                files_changed = [fname for fname, _ in commit.stats.files.items()]

                all_security_commits.append({
                    "repo": repo_name,
                    "commit_hash": commit.hexsha,
                    "message": msg,
                    "author": str(commit.author),
                    "date": str(commit.committed_datetime),
                    "files": files_changed
                })
    except Exception as e:
        print(f"[ERROR] {repo_name}: {e}")

# === SAVE TO FILE ===
with open("nasa_security_commits_up.json", "w") as f:
    json.dump(all_security_commits, f, indent=2)

print(f"\n Extracted {len(all_security_commits)} security-related commits from NASA repos.")


In [ ]:
import json
import csv

# Load the JSON data
with open("nasa_security_commits_up.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Define output CSV file
output_csv = "nasa_security_commits_up.csv"

# Get all fields (flatten list if needed)
fieldnames = ["repo", "commit_hash", "message", "author", "date", "files"]

# Write to CSV
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    for entry in data:
        # Convert file list to comma-separated string
        entry["files"] = ",".join(entry.get("files", []))
        writer.writerow(entry)

print(f" Converted to CSV: {output_csv}")

In [ ]:
import os
import re
import hashlib
import pandas as pd
import requests
import json
import subprocess
from git import Repo, GitCommandError
from clang import cindex

# === CONFIG ===
cindex.Config.set_library_file(r"C:/Program Files/LLVM/bin/libclang.dll")  # Adjust to your system
CSV_PATH = "nasa_security_commits_up.csv"
REPO_DIR = "./nasa_repos"
CVE_REGEX = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_RE = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cve/1.0/"
SEMGREP_RULES_PATH = "p/cwe-top-25"

# === HELPERS ===
def extract_function_clang(file_path, target_line):
    index = cindex.Index.create()
    try:
        tu = index.parse(file_path, args=['-x', 'c++'])
        for node in tu.cursor.walk_preorder():
            if node.kind == cindex.CursorKind.FUNCTION_DECL:
                extent = node.extent
                if extent.start.line <= target_line <= extent.end.line:
                    return get_source_segment(extent)
    except Exception as e:
        print("    Clang parsing failed:", e)
    return ""

def get_source_segment(extent):
    try:
        with open(extent.start.file.name, encoding='utf-8') as f:
            lines = f.readlines()
        return "".join(lines[extent.start.line - 1:extent.end.line])
    except:
        return ""

def md5_hash(code: str):
    return hashlib.md5(code.encode('utf-8')).hexdigest()

def fetch_cwe_from_cve(cve_id):
    try:
        response = requests.get(NVD_API_URL + cve_id)
        if response.status_code == 200:
            items = response.json()["result"]["CVE_Items"]
            if items:
                cwes = items[0]['cve']['problemtype']['problemtype_data'][0]['description']
                return [entry['value'] for entry in cwes if entry['value'] != 'NVD-CWE-Other']
    except:
        pass
    return []

def run_semgrep_on_code(code: str, rule_path: str):
    tmp_file = "tmp/semgrep_target.c"
    with open(tmp_file, "w", encoding="utf-8") as f:
        f.write(code)
    try:
        result = subprocess.run([
        "semgrep", "--config", rule_path, tmp_file, "--json"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, encoding='utf-8')


        findings = json.loads(result.stdout).get("results", [])
        cwe_ids = set()
        for finding in findings:
            metadata = finding.get("extra", {}).get("metadata", {})
            cwe = metadata.get("cwe")
            if cwe:
                cwe_ids.add(cwe)
        return list(cwe_ids)
    except subprocess.CalledProcessError as e:
        print("     Semgrep error:", e)
    return []

# === DEBUG SAMPLE RUN ===
df = pd.read_csv(CSV_PATH).dropna(subset=["files"]).head(500)
results = []
cve_to_cwe = {}

tmp_old_file = "tmp/old_temp.c"
tmp_new_file = "tmp/new_temp.c"
os.makedirs("tmp", exist_ok=True)

for idx, row in df.iterrows():
    repo_name, sha, msg = row["repo"], row["commit_hash"], row["message"]
    repo_path = os.path.join(REPO_DIR, repo_name)
    if not os.path.isdir(repo_path):
        continue

    print(f"\n Debugging {repo_name}@{sha[:7]}")

    try:
        repo = Repo(repo_path)
        commit = repo.commit(sha)
        parent = commit.parents[0] if commit.parents else None
        if not parent:
            print("     No parent commit, skipping.")
            continue

        diff = commit.diff(parent, create_patch=True)
        for d in diff:
            path = d.a_path
            if not path or not path.endswith((".c", ".cpp", ".cc", ".cxx")):
                continue

            print(f"   File: {path}")
            try:
                old_code = repo.git.show(f"{parent}:{path}")
                new_code = repo.git.show(f"{sha}:{path}")
            except Exception as e:
                print(f"     File access error: {e}")
                continue

            with open(tmp_old_file, "w", encoding="utf-8") as f: f.write(old_code)
            with open(tmp_new_file, "w", encoding="utf-8") as f: f.write(new_code)

            patch = d.diff.decode(errors="ignore").splitlines()
            hunk_lines = [(int(m.group(1)), int(m.group(2))) for m in map(HUNK_RE.match, patch) if m]
            if not hunk_lines:
                print("      No hunk lines found")
                continue

            for old_line, new_line in hunk_lines:
                print(f"   Hunk lines → old: {old_line}, new: {new_line}")

                old_fn = extract_function_clang(tmp_old_file, old_line)
                new_fn = extract_function_clang(tmp_new_file, new_line)

                if old_fn:
                    print(f"    OLD function (hash {md5_hash(old_fn)[:8]}):\n{old_fn.strip()[:200]}\n...")
                if new_fn:
                    print(f"    NEW function (hash {md5_hash(new_fn)[:8]}):\n{new_fn.strip()[:200]}\n...")

                cve_match = CVE_REGEX.search(msg)
                cve_id = cve_match.group(1) if cve_match else None

                cwe_ids = []
                if cve_id:
                    if cve_id not in cve_to_cwe:
                        cve_to_cwe[cve_id] = fetch_cwe_from_cve(cve_id)
                    cwe_ids = cve_to_cwe[cve_id]

                # Use Semgrep as fallback if no CWE ID found from CVE
                if not cwe_ids and old_fn:
                    cwe_ids = run_semgrep_on_code(old_fn, SEMGREP_RULES_PATH)

                if old_fn.strip() or new_fn.strip():
                    results.append({
                        "repo": repo_name,
                        "commit": sha,
                        "file": path,
                        "cve_id": cve_id,
                        "cwe_ids": ",".join(cwe_ids),
                        "old_line": old_line,
                        "new_line": new_line,
                        "old_md5": md5_hash(old_fn) if old_fn else "",
                        "new_md5": md5_hash(new_fn) if new_fn else "",
                        "old_func": old_fn.strip() if old_fn else "",
                        "new_func": new_fn.strip() if new_fn else ""
                    })

    except Exception as e:
        print(f"    Commit-level error: {e}")
        continue


out_df = pd.DataFrame(results)
out_df.to_csv("vuln_funcs_with_cwe_new.csv", index=False)
print(f"\nSaved {len(out_df)} function-level entries to vuln_funcs_with_cwe.csv")


In [ ]:
import pandas as pd

data = pd.read_csv("vuln_funcs_with_cwe_new.csv")
data.head()

In [ ]:
data.info()

In [ ]:
import pandas as pd
import subprocess
import json
import os

# === CONFIG ===
CSV_PATH = "vuln_funcs_with_cwe_new.csv"
SEMGREP_RULES_PATH = "p/cwe-top-25"
TMP_FILE = "tmp/semgrep_code.c"
os.makedirs("tmp", exist_ok=True)

# === Function: Run Semgrep on Code String ===
def run_semgrep_on_code(code: str, rule_path: str):
    with open(TMP_FILE, "w", encoding="utf-8") as f:
        f.write(code)
    try:
        result = subprocess.run([
            "semgrep", "--config", rule_path, TMP_FILE, "--json"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, encoding='utf-8')

        findings = json.loads(result.stdout).get("results", [])
        cwe_ids = set()
        for finding in findings:
            metadata = finding.get("extra", {}).get("metadata", {})
            cwe = metadata.get("cwe")
            if cwe:
                cwe_ids.add(cwe)
        return ",".join(sorted(cwe_ids))
    except subprocess.CalledProcessError as e:
        print("Semgrep error:", e.stderr)
        return ""
    except Exception as e:
        print("Unexpected error:", e)
        return ""

# === Load Dataset ===
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["old_func"])

# === Apply Semgrep ===
df["cwe_ids"] = df["old_func"].apply(lambda code: run_semgrep_on_code(code, SEMGREP_RULES_PATH))

# === Save Output ===
df.to_csv("vuln_funcs_with_semgrep_cwe.csv", index=False)
print(f"Updated file saved as 'vuln_funcs_with_semgrep_cwe.csv' with {len(df)} entries.")


In [ ]:
df = pd.read_csv("vuln_funcs_with_semgrep_cwe.csv")
df.info()

In [ ]:
import os
from git import Repo
import re
import json
from tqdm import tqdm

# === CONFIG ===
LOCAL_REPO_DIR = "./nasa_repos"
CVE_REGEX = re.compile(r'\bCVE-\d{4}-\d{4,7}\b', re.IGNORECASE)

# === HELPER FUNCTION ===
def is_cve_commit(message):
    return bool(CVE_REGEX.search(message))

# === PROCESS COMMITS IN EACH REPO ===
all_cve_commits = []

for repo_name in tqdm(os.listdir(LOCAL_REPO_DIR), desc="Scanning Repos"):
    repo_path = os.path.join(LOCAL_REPO_DIR, repo_name)
    if not os.path.isdir(repo_path):
        continue
    try:
        repo = Repo(repo_path)
        for commit in repo.iter_commits():
            msg = commit.message
            if is_cve_commit(msg):
                files_changed = [fname for fname, _ in commit.stats.files.items()]

                all_cve_commits.append({
                    "repo": repo_name,
                    "commit_hash": commit.hexsha,
                    "message": msg,
                    "author": str(commit.author),
                    "date": str(commit.committed_datetime),
                    "files": files_changed
                })
    except Exception as e:
        print(f"[ERROR] {repo_name}: {e}")

# === SAVE TO FILE ===
with open("nasa_cve_commits.json", "w") as f:
    json.dump(all_cve_commits, f, indent=2)

print(f"\nExtracted {len(all_cve_commits)} commits with CVE mentions.")


In [ ]:
import os
from git import Repo
import re
import json
from tqdm import tqdm

# === CONFIG ===
LOCAL_REPO_DIR = "./nasa_repos"
# Looser regex to match CVE-2023-1234, cve_2023_456, CVE 2021 0001, etc.
CVE_REGEX = re.compile(r'\bCVE[-_ ]?\d{4}[-_ ]?\d{3,7}\b', re.IGNORECASE)

# === HELPER FUNCTIONS ===
def extract_cve_ids(message):
    return CVE_REGEX.findall(message)

# === PROCESS COMMITS IN EACH REPO ===
all_cve_commits = []

for repo_name in tqdm(os.listdir(LOCAL_REPO_DIR), desc="Scanning Repos"):
    repo_path = os.path.join(LOCAL_REPO_DIR, repo_name)
    if not os.path.isdir(repo_path):
        continue
    try:
        repo = Repo(repo_path)
        for commit in repo.iter_commits():
            msg = commit.message
            cve_ids = extract_cve_ids(msg)

            # Debug: print messages containing "cve" even if not matched
            if "cve" in msg.lower() and not cve_ids:
                print("[NO MATCH] Possible CVE:", msg.strip())

            if cve_ids:
                files_changed = [fname for fname, _ in commit.stats.files.items()]

                all_cve_commits.append({
                    "repo": repo_name,
                    "commit_hash": commit.hexsha,
                    "message": msg.strip(),
                    "author": str(commit.author),
                    "date": str(commit.committed_datetime),
                    "files": files_changed,
                    "cve_ids": cve_ids
                })
    except Exception as e:
        print(f"[ERROR] {repo_name}: {e}")

# === SAVE TO FILE ===
with open("nasa_cve_commits.json", "w") as f:
    json.dump(all_cve_commits, f, indent=2)

print(f"\nExtracted {len(all_cve_commits)} commits mentioning CVEs.")


In [ ]:
ds = pd.read_csv('nasa_security_commits_up.csv')

In [ ]:
import pandas as pd
import re

# Step 1: Load your DataFrame if not already
# df = pd.read_csv("your_file.csv")  # or use JSON, etc.

# Step 2: Define a loose regex for CVE detection
CVE_REGEX = re.compile(r'\bCVE[-_ ]?\d{4}[-_ ]?\d{3,7}\b', re.IGNORECASE)

# Step 3: Extract CVE ID from the commit message
def extract_cve_from_text(text):
    if pd.isna(text):
        return None
    matches = CVE_REGEX.findall(text)
    return matches[0] if matches else None

# Step 4: Apply this function to the column with commit messages
# Replace 'message' below with the actual name of your message column
ds['cve_id'] = ds['message'].apply(extract_cve_from_text)

# Step 5: Check results
print(ds['cve_id'].value_counts(dropna=False).head(10))


In [ ]:
import pandas as pd
import re

# Define regex for buffer overflow (flexible: matches "buffer overflow", "buffer-overflow", etc.)
BUFFER_OVERFLOW_REGEX = re.compile(r'\bbuffer[\s_-]?overflow\b', re.IGNORECASE)

# Function to extract buffer overflow mention
def extract_buffer_overflow(text):
    if pd.isna(text):
        return None
    return "buffer_overflow" if BUFFER_OVERFLOW_REGEX.search(text) else None

# Apply this to your commit message column
# Replace 'message' with your actual commit message column name
ds['buffer_overflow'] = ds['message'].apply(extract_buffer_overflow)

# Show frequency of buffer overflow mentions
print(ds['buffer_overflow'].value_counts(dropna=False))


In [ ]:
import pandas as pd
6import re

# Your list of security-related keywords
SECURITY_KEYWORDS = [
    "cve", "cwe", "security", "vulnerability", "vulnerable",
    "buffer overflow", "fix", "patch"
]

# Initialize a dictionary to store keyword counts
keyword_counts = {kw: 0 for kw in SECURITY_KEYWORDS}

# Column with commit messages (update if it's named differently)
message_column = 'message'

# Count keyword occurrences in each message
for msg in ds[message_column].dropna():
    msg_lower = msg.lower()
    for kw in SECURITY_KEYWORDS:
        if kw in msg_lower:
            keyword_counts[kw] += 1



keyword_df = pd.DataFrame(list(keyword_counts.items()), columns=['Keyword', 'Count']).sort_values(by='Count', ascending=False)

# Show the counts
print(keyword_df)



In [ ]:
import pandas as pd
import re

# === Step 1: Load your dataset ===
  # Replace with your actual file name

# === Step 2: Define function completeness checker (no clang) ===
def is_complete_c_function(code):
    # Normalize whitespace and strip leading/trailing junk
    code = code.strip()

    # Check for valid function signature (e.g., int foo(), void bar(int x), etc.)
    has_signature = bool(re.search(
        r'\b(?:void|int|char|float|double|bool|long|short)\s+\**\w+\s*\([^)]*\)\s*{',
        code
    ))

    # Check that code ends with a closing brace
    has_closing_brace = code.endswith('}')

    # Check that there are balanced curly braces
    open_braces = code.count('{')
    close_braces = code.count('}')

    return has_signature and has_closing_brace and open_braces == close_braces

# === Step 3: Apply to your dataset ===
data["is_complete_function"] = data["old_func"].apply(is_complete_c_function)

# === Step 4: Show results (optional) ===
print(" Completed function detection.")
display(data[["old_func", "is_complete_function"]].head())

In [ ]:
data["is_complete_function"].value_counts()

In [ ]:
import os
import re
import hashlib
import pandas as pd
import requests
import json
import subprocess
from git import Repo
from clang import cindex
from multiprocessing import Pool, cpu_count

# === CONFIG ===
#cindex.Config.set_library_file(r"C:/Program Files/LLVM/bin/libclang.dll")
CSV_PATH = "nasa_security_commits_up.csv"
REPO_DIR = "./nasa_repos"
SEMGREP_RULES_PATH = "p/cwe-top-25"
CVE_REGEX = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_RE = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cve/1.0/"
TMP_DIR = "tmp"
os.makedirs(TMP_DIR, exist_ok=True)

def extract_function_clang(file_path, target_line):
    index = cindex.Index.create()
    try:
        tu = index.parse(file_path, args=['-x', 'c++'])
        for node in tu.cursor.walk_preorder():
            if node.kind == cindex.CursorKind.FUNCTION_DECL:
                extent = node.extent
                if extent.start.line <= target_line <= extent.end.line:
                    return get_source_segment(extent)
    except:
        pass
    return ""

def get_source_segment(extent):
    try:
        with open(extent.start.file.name, encoding='utf-8') as f:
            lines = f.readlines()
        return "".join(lines[extent.start.line - 1:extent.end.line])
    except:
        return ""

def md5_hash(code: str):
    return hashlib.md5(code.encode('utf-8')).hexdigest()

def fetch_cwe_from_cve(cve_id):
    try:
        response = requests.get(NVD_API_URL + cve_id)
        if response.status_code == 200:
            items = response.json()["result"]["CVE_Items"]
            if items:
                cwes = items[0]['cve']['problemtype']['problemtype_data'][0]['description']
                return [entry['value'] for entry in cwes if entry['value'] != 'NVD-CWE-Other']
    except:
        pass
    return []

def run_semgrep_on_code(code: str, rule_path: str):
    tmp_file = os.path.join(TMP_DIR, f"semgrep_{os.getpid()}.c")
    with open(tmp_file, "w", encoding="utf-8") as f:
        f.write(code)
    try:
        result = subprocess.run([
            "semgrep", "--config", rule_path, tmp_file, "--json"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, encoding='utf-8')
        findings = json.loads(result.stdout).get("results", [])
        cwe_ids = set()
        for finding in findings:
            metadata = finding.get("extra", {}).get("metadata", {})
            cwe = metadata.get("cwe")
            if cwe:
                cwe_ids.add(cwe)
        return list(cwe_ids)
    except subprocess.CalledProcessError:
        return []

def process_commit(row_dict):
    result_rows = []
    repo_name, sha, msg = row_dict["repo"], row_dict["commit_hash"], row_dict["message"]
    repo_path = os.path.join(REPO_DIR, repo_name)
    tmp_old_file = os.path.join(TMP_DIR, f"old_temp_{os.getpid()}.c")
    tmp_new_file = os.path.join(TMP_DIR, f"new_temp_{os.getpid()}.c")

    try:
        if not os.path.isdir(repo_path):
            return []
        repo = Repo(repo_path)
        commit = repo.commit(sha)
        parent = commit.parents[0] if commit.parents else None
        if not parent:
            return []

        diff = commit.diff(parent, create_patch=True)
        for d in diff:
            path = d.a_path
            if not path or not path.endswith((".c", ".cpp", ".cc", ".cxx")):
                continue

            try:
                old_code = repo.git.show(f"{parent}:{path}")
                new_code = repo.git.show(f"{sha}:{path}")
            except:
                continue

            with open(tmp_old_file, "w", encoding="utf-8") as f: f.write(old_code)
            with open(tmp_new_file, "w", encoding="utf-8") as f: f.write(new_code)

            patch = d.diff.decode(errors="ignore").splitlines()
            hunk_lines = [(int(m.group(1)), int(m.group(2))) for m in map(HUNK_RE.match, patch) if m]
            if not hunk_lines:
                continue

            for old_line, new_line in hunk_lines:
                old_fn = extract_function_clang(tmp_old_file, old_line)
                new_fn = extract_function_clang(tmp_new_file, new_line)

                cve_match = CVE_REGEX.search(msg)
                cve_id = cve_match.group(1) if cve_match else None

                cwe_ids = []
                if cve_id:
                    cwe_ids = fetch_cwe_from_cve(cve_id)

                if not cwe_ids and old_fn:
                    cwe_ids = run_semgrep_on_code(old_fn, SEMGREP_RULES_PATH)

                if old_fn.strip() or new_fn.strip():
                    result_rows.append({
                        "repo": repo_name,
                        "commit": sha,
                        "file": path,
                        "cve_id": cve_id,
                        "cwe_ids": ",".join(cwe_ids),
                        "old_line": old_line,
                        "new_line": new_line,
                        "old_md5": md5_hash(old_fn) if old_fn else "",
                        "new_md5": md5_hash(new_fn) if new_fn else "",
                        "old_func": old_fn.strip() if old_fn else "",
                        "new_func": new_fn.strip() if new_fn else ""
                    })
    except:
        return []
    return result_rows

# === Load and slice dataframe (next 500)
df = pd.read_csv(CSV_PATH).dropna(subset=["files"]).iloc[500:1000]
rows = df.to_dict(orient="records")

# === Parallel processing
with Pool(cpu_count()) as pool:
    all_results = pool.map(process_commit, rows)

# === Flatten and save
flattened = [item for sublist in all_results if sublist for item in sublist]
out_df = pd.DataFrame(flattened)
out_df.to_csv("vuln_funcs_with_cwe_new_batch2.csv", index=False)

print(f"\n Saved {len(out_df)} function-level entries to vuln_funcs_with_cwe_new_batch2.csv")


In [ ]:
import os
import re
import hashlib
import pandas as pd
import requests
import json
import subprocess
from git import Repo, GitCommandError
from clang import cindex

# === CONFIG ===
cindex.Config.set_library_file(r"C:/Program Files/LLVM/bin/libclang.dll")  # Adjust to your system
CSV_PATH = "nasa_security_commits_up.csv"
REPO_DIR = "./nasa_repos"
CVE_REGEX = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_RE = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cve/1.0/"
SEMGREP_RULES_PATH = "p/cwe-top-25"

# === HELPERS ===
def extract_function_clang(file_path, target_line):
    index = cindex.Index.create()
    try:
        tu = index.parse(file_path, args=['-x', 'c++'])
        for node in tu.cursor.walk_preorder():
            if node.kind == cindex.CursorKind.FUNCTION_DECL:
                extent = node.extent
                if extent.start.line <= target_line <= extent.end.line:
                    return get_source_segment(extent)
    except Exception as e:
        print("    Clang parsing failed:", e)
    return ""

def get_source_segment(extent):
    try:
        with open(extent.start.file.name, encoding='utf-8') as f:
            lines = f.readlines()
        return "".join(lines[extent.start.line - 1:extent.end.line])
    except:
        return ""

def md5_hash(code: str):
    return hashlib.md5(code.encode('utf-8')).hexdigest()

def fetch_cwe_from_cve(cve_id):
    try:
        response = requests.get(NVD_API_URL + cve_id)
        if response.status_code == 200:
            items = response.json()["result"]["CVE_Items"]
            if items:
                cwes = items[0]['cve']['problemtype']['problemtype_data'][0]['description']
                return [entry['value'] for entry in cwes if entry['value'] != 'NVD-CWE-Other']
    except:
        pass
    return []

def run_semgrep_on_code(code: str, rule_path: str):
    tmp_file = "tmp/semgrep_target.c"
    with open(tmp_file, "w", encoding="utf-8") as f:
        f.write(code)
    try:
        result = subprocess.run([
        "semgrep", "--config", rule_path, tmp_file, "--json"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, encoding='utf-8')


        findings = json.loads(result.stdout).get("results", [])
        cwe_ids = set()
        for finding in findings:
            metadata = finding.get("extra", {}).get("metadata", {})
            cwe = metadata.get("cwe")
            if cwe:
                cwe_ids.add(cwe)
        return list(cwe_ids)
    except subprocess.CalledProcessError as e:
        print("     Semgrep error:", e)
    return []

# === DEBUG SAMPLE RUN ===
df = pd.read_csv(CSV_PATH).dropna(subset=["files"]).iloc[500:1000]
results = []
cve_to_cwe = {}

tmp_old_file = "tmp/old_temp.c"
tmp_new_file = "tmp/new_temp.c"
os.makedirs("tmp", exist_ok=True)

for idx, row in df.iterrows():
    repo_name, sha, msg = row["repo"], row["commit_hash"], row["message"]
    repo_path = os.path.join(REPO_DIR, repo_name)
    if not os.path.isdir(repo_path):
        continue

    print(f"\n Debugging {repo_name}@{sha[:7]}")

    try:
        repo = Repo(repo_path)
        commit = repo.commit(sha)
        parent = commit.parents[0] if commit.parents else None
        if not parent:
            print("     No parent commit, skipping.")
            continue

        diff = commit.diff(parent, create_patch=True)
        for d in diff:
            path = d.a_path
            if not path or not path.endswith((".c", ".cpp", ".cc", ".cxx")):
                continue

            print(f"   File: {path}")
            try:
                old_code = repo.git.show(f"{parent}:{path}")
                new_code = repo.git.show(f"{sha}:{path}")
            except Exception as e:
                print(f"     File access error: {e}")
                continue

            with open(tmp_old_file, "w", encoding="utf-8") as f: f.write(old_code)
            with open(tmp_new_file, "w", encoding="utf-8") as f: f.write(new_code)

            patch = d.diff.decode(errors="ignore").splitlines()
            hunk_lines = [(int(m.group(1)), int(m.group(2))) for m in map(HUNK_RE.match, patch) if m]
            if not hunk_lines:
                print("      No hunk lines found")
                continue

            for old_line, new_line in hunk_lines:
                print(f"   Hunk lines → old: {old_line}, new: {new_line}")

                old_fn = extract_function_clang(tmp_old_file, old_line)
                new_fn = extract_function_clang(tmp_new_file, new_line)

                if old_fn:
                    print(f"    OLD function (hash {md5_hash(old_fn)[:8]}):\n{old_fn.strip()[:200]}\n...")
                if new_fn:
                    print(f"    NEW function (hash {md5_hash(new_fn)[:8]}):\n{new_fn.strip()[:200]}\n...")

                cve_match = CVE_REGEX.search(msg)
                cve_id = cve_match.group(1) if cve_match else None

                cwe_ids = []
                if cve_id:
                    if cve_id not in cve_to_cwe:
                        cve_to_cwe[cve_id] = fetch_cwe_from_cve(cve_id)
                    cwe_ids = cve_to_cwe[cve_id]

                # Use Semgrep as fallback if no CWE ID found from CVE
                if not cwe_ids and old_fn:
                    cwe_ids = run_semgrep_on_code(old_fn, SEMGREP_RULES_PATH)

                if old_fn.strip() or new_fn.strip():
                    results.append({
                        "repo": repo_name,
                        "commit": sha,
                        "file": path,
                        "cve_id": cve_id,
                        "cwe_ids": ",".join(cwe_ids),
                        "old_line": old_line,
                        "new_line": new_line,
                        "old_md5": md5_hash(old_fn) if old_fn else "",
                        "new_md5": md5_hash(new_fn) if new_fn else "",
                        "old_func": old_fn.strip() if old_fn else "",
                        "new_func": new_fn.strip() if new_fn else ""
                    })

    except Exception as e:
        print(f"    Commit-level error: {e}")
        continue


out_df = pd.DataFrame(results)
out_df.to_csv("vuln_funcs_with_cwe_new.csv", index=False)
print(f"\nSaved {len(out_df)} function-level entries to vuln_funcs_with_cwe.csv")

In [ ]:
out_df.to_csv("vuln_funcs_with_cwe_new_batch2.csv", index=False, quoting=1)


In [ ]:
import pandas as pd

ddd= pd.read_csv("vuln_funcs_with_cwe_new_batch2.csv")
ddd.info()

In [ ]:
import pandas as pd
import re

# === Step 1: Load your dataset ===
  # Replace with your actual file name

# === Step 2: Define function completeness checker (no clang) ===
import re

def is_complete_c_function(code):
    if not isinstance(code, str):
        return False

    code = code.strip()

    # Normalize and remove comment blocks
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)  # block comments
    code = re.sub(r'//.*', '', code)  # single-line comments

    # Remove strings to avoid brace confusion
    code = re.sub(r'"(?:\\.|[^"\\])*"', '""', code)  # string literals
    code = re.sub(r"'(?:\\.|[^'\\])+'", "''", code)  # char literals

    # Regex for common return types and modifiers
    signature_pattern = re.compile(
        r"""
        ^                                  # start of string
        (\s*(static|inline|extern)?\s*)?   # optional storage class
        (\s*(unsigned|signed|long|short|const)?\s*)*   # optional qualifiers
        \b(void|int|char|float|double|bool)\b  # return type
        (\s+\*?\s*\w+)\s*                   # function name
        \([^)]*\)\s*                        # argument list
        \{                                  # opening brace
        """, re.VERBOSE
    )

    has_signature = bool(signature_pattern.search(code))

    # Count opening and closing braces (excluding inside strings/comments)
    open_braces = code.count('{')
    close_braces = code.count('}')
    balanced_braces = open_braces == close_braces

    # Final closing brace
    ends_with_brace = code.strip().endswith('}')

    return has_signature and balanced_braces and ends_with_brace

# === Step 3: Apply to your dataset ===
ddd["is_complete_function"] = ddd["old_func"].apply(is_complete_c_function)

# === Step 4: Show results (optional) ===
print(" Completed function detection.")
display(ddd[["old_func", "is_complete_function"]].head())

In [ ]:
ddd["is_complete_function"].value_counts()

In [ ]:
import os
import pandas as pd
import subprocess
import json

# === CONFIG ===
CSV_PATH = "vuln_funcs_with_cwe_new.csv"  # Use your actual file
SEMGREP_RULES_PATH = "p/default"  # Broad rule coverage
TMP_DIR = "tmp_test_funcs"
os.makedirs(TMP_DIR, exist_ok=True)

# === Load First 100 Functions
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["old_func"]).head(100)

# === Run Semgrep on Each Function
for idx, row in df.iterrows():
    code = row["old_func"]
    file_path = os.path.join(TMP_DIR, f"func_{idx}.c")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(code)

    print(f" [Function #{idx}] Running Semgrep on func_{idx}.c")

    try:
        result = subprocess.run([
            "semgrep", "--config", SEMGREP_RULES_PATH, file_path, "--json"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, encoding='utf-8')

        findings = json.loads(result.stdout).get("results", [])
        if findings:
            print(f" Found {len(findings)} finding(s):")
            for finding in findings:
                rule_id = finding.get("check_id", "")
                message = finding.get("extra", {}).get("message", "")
                cwe = finding.get("extra", {}).get("metadata", {}).get("cwe", "N/A")
                print(f" - Rule: {rule_id}, CWE: {cwe}, Message: {message}")
        else:
            print(" No findings")

    except subprocess.CalledProcessError as e:
        print(" Semgrep error:", e.stderr)


In [ ]:


# === Step 4: Save complete function dataset ===
complete_df = ddd[ddd["is_complete_function"] == True]
complete_df.to_csv("complete_functions_only.csv", index=False,quoting=1)

print(f" Extracted {len(complete_df)} complete functions.")


In [ ]:
import pandas as pd
import os

# === CONFIG ===
CSV_PATH = "complete_functions_only.csv"
COLUMN_NAME = "old_func"
OUTPUT_DIR = "codeql_c_funcs"

# === Load Dataset ===
df = pd.read_csv(CSV_PATH)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Write Each C Function to .c File ===
for idx, code in enumerate(df[COLUMN_NAME]):
    filename = f"fun_{idx}.c"
    file_path = os.path.join(OUTPUT_DIR, filename)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"// FunctionID: {idx}\n{code.strip()}")


In [ ]:
import pandas as pd

devign = pd.read_json("function.json")
devign.info()

In [ ]:
import pandas as pd
train_df = pd.read_csv('train_df.csv')
test_df = pd.read_csv('test_df.csv')

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import torch

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer,
    TrainingArguments, DataCollatorWithPadding
)


# === HuggingFace Dataset ===
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.rename_column("target", "labels")
test_dataset = test_dataset.rename_column("target", "labels")

model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, use_safetensors=True)
data_collator = DataCollatorWithPadding(tokenizer)

def tokenize_fn(ex): return tokenizer(ex["func"], truncation=True, padding="max_length", max_length=512)
train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# === Custom metrics ===
def compute_metrics(p):
    preds = torch.argmax(torch.tensor(p.predictions), axis=1).numpy()
    labels = p.label_ids

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    mcc = matthews_corrcoef(labels, preds)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel() if len(set(labels)) == 2 else (0, 0, 0, 0)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "mcc": mcc,
        "sensitivity": sensitivity,
        "specificity": specificity
    }

# === Training setup ===
training_args = TrainingArguments(
    output_dir="./codebert_devign_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# === Train and save ===
trainer.train()
trainer.save_model("./codebert_devign_model")
tokenizer.save_pretrained("./codebert_devign_model")

In [ ]:
ds_f= pd.read_csv('nasa_first_half all.csv')
ds_s = pd.read_csv('nasa_second_half 400.csv')

In [ ]:
ds_f.info()

In [ ]:
import pandas as pd

import re

import re

def is_meaningful_function(code, min_statements=2):
    if not isinstance(code, str):
        return False

    code = code.strip()

    if '{' not in code or '}' not in code:
        return False

    # Extract body
    try:
        open_idx = code.index('{')
        close_idx = code.rindex('}')
        body = code[open_idx + 1 : close_idx]
    except ValueError:
        return False

    # Remove block and inline comments
    body = re.sub(r'/\*.*?\*/', '', body, flags=re.DOTALL)
    body = re.sub(r'//.*', '', body)

    # Count statements (lines ending in semicolon)
    statement_count = body.count(';')

    return statement_count > min_statements


# === Load your DataFrame ===
# df = pd.read_csv("your_file.csv")  # already loaded as 'ddd' maybe?

# Apply filter
filtered_df = ds_f[ds_f["Code"].apply(is_meaningful_function)].copy()

print(f" Filtered: {len(ds_f) - len(filtered_df)} rows removed, {len(filtered_df)} remain.")


In [ ]:
import pandas as pd

import re

import re

def is_meaningful_function(code, min_statements=2):
    if not isinstance(code, str):
        return False

    code = code.strip()

    if '{' not in code or '}' not in code:
        return False

    # Extract body
    try:
        open_idx = code.index('{')
        close_idx = code.rindex('}')
        body = code[open_idx + 1 : close_idx]
    except ValueError:
        return False

    # Remove block and inline comments
    body = re.sub(r'/\*.*?\*/', '', body, flags=re.DOTALL)
    body = re.sub(r'//.*', '', body)

    # Count statements (lines ending in semicolon)
    statement_count = body.count(';')

    return statement_count > min_statements


# === Load your DataFrame ===
# df = pd.read_csv("your_file.csv")  # already loaded as 'ddd' maybe?

# Apply filter
filtered_df_s = ds_s[ds_f["Code"].apply(is_meaningful_function)].copy()

print(f" Filtered: {len(ds_s)- len(filtered_df_s)} rows removed, {len(filtered_df_s)} remain.")

In [ ]:
ds_f["lines_of_code"] = ds_f["Code"].apply(lambda x: len([l.strip() for l in x.split('\n') if l.strip()]) if isinstance(x, str) else 0)
ds_f["valid_func"] = ds_f["Code"].apply(is_meaningful_function)

# Example: Show what's being dropped
print(ds_f[~ds_f["valid_func"]][["Code", "lines_of_code"]].head())


In [ ]:
filtered_df.to_csv("nasa_first_half_meaningful.csv", index=False,quoting=1)

In [ ]:
filtered_df_s.to_csv("nasa_second_half_meaningful.csv", index=False,quoting=1)

In [ ]:
nasa_df = pd.concat([filtered_df, filtered_df_s], axis=0, ignore_index=True)

nasa_df.info()

In [ ]:
nasa_df.drop(["lines_of_code", "valid_func"], axis=1, inplace=True)


In [ ]:
nasa_df.info()

In [ ]:
nasa_df.to_csv("nasa_data_vul.csv", index=False,quoting=1)

In [ ]:
df = pd.read_csv("complete_functions_only.csv")
df_dedup = df.drop_duplicates(subset="old_func", keep="first").reset_index(drop=True)
df_dedup.info()

In [ ]:
df_dedup.to_csv("nasa_complete_func_deduplicated.csv", index=False,quoting=1)

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer,
    TrainingArguments, DataCollatorWithPadding
)


# === HuggingFace Dataset ===
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.rename_column("target", "labels")
test_dataset = test_dataset.rename_column("target", "labels")

model_name = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer)

def tokenize_fn(ex): return tokenizer(ex["func"], truncation=True, padding="max_length", max_length=512)
train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# === Custom metrics ===
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr":         fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1":          f1_score(labels, preds, zero_division=0),
        "mcc":         matthews_corrcoef(labels, preds),
        "kappa":       cohen_kappa_score(labels, preds),
        "mse":         mean_squared_error(labels, preds),
        "mae":         mean_absolute_error(labels, preds),
        "auc":         auc
    }

# === Training setup ===
training_args = TrainingArguments(
    output_dir="./graphcodebert_devign_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# === Train and save ===
trainer.train()
trainer.save_model("./graphcodebert_devign_model")
tokenizer.save_pretrained("./graphcodebert_devign_model")

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)

# === Load new test data ===
new_test_df = pd.read_csv("test_df.csv")  # Must contain 'func' and 'target'
new_test_df = new_test_df.dropna(subset=["func", "target"])  # Clean
new_test_df = new_test_df.rename(columns={"target": "labels"})

# === Load saved model and tokenizer ===
model_path = "./graphcodebert_devign_model_UP"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# === Tokenize and create Dataset ===
new_dataset = Dataset.from_pandas(new_test_df)
new_dataset = new_dataset.map(lambda x: tokenizer(x["func"], truncation=True, padding="max_length", max_length=512), batched=True)

# === Metrics ===
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr":         fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1":          f1_score(labels, preds, zero_division=0),
        "mcc":         matthews_corrcoef(labels, preds),
        "kappa":       cohen_kappa_score(labels, preds),
        "mse":         mean_squared_error(labels, preds),
        "mae":         mean_absolute_error(labels, preds),
        "auc":         auc
    }

# === Evaluate using Trainer ===
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

metrics = trainer.evaluate(new_dataset)
print(" Evaluation Metrics on New Data:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


In [ ]:
torch.cuda.is_available()


In [ ]:
train_df.info()

In [ ]:
train_df['target'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)
from collections import Counter

train_df = pd.read_csv("train_df.csv")
test_df = pd.read_csv("test_df.csv")

train_df["target"] = train_df["target"].astype(int)
test_df["target"] = test_df["target"].astype(int)

# === HuggingFace Datasets ===
train_dataset = Dataset.from_pandas(train_df.rename(columns={"target": "labels"}))
test_dataset = Dataset.from_pandas(test_df.rename(columns={"target": "labels"}))

# === Tokenizer & Tokenization ===
model_name = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_fn(example):
    return tokenizer(example["func"], truncation=True, padding="max_length", max_length=512)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# === Model (with safety for classification head) ===
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)
model.config.problem_type = "single_label_classification"


# === Custom metrics ===
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    except:
        tn = fp = fn = tp = 0

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "kappa": cohen_kappa_score(labels, preds),
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "auc": auc
    }

# === Training Arguments ===
training_args = TrainingArguments(
    output_dir="./graphcodebert_devign_model_UP",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16 =True,
    report_to="none"
)

# === Trainer ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# === Train and Save ===
trainer.train()
trainer.save_model("./graphcodebert_devign_model_UP")
tokenizer.save_pretrained("./graphcodebert_devign_model_UP")

# === Optional: Print class balance
print("Train label distribution:", Counter(train_dataset["labels"]))
print("Test label distribution:", Counter(test_dataset["labels"]))


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)
from collections import Counter

train_df = pd.read_csv("train_df.csv")
test_df = pd.read_csv("test_df.csv")

train_df["target"] = train_df["target"].astype(int)
test_df["target"] = test_df["target"].astype(int)

# === HuggingFace Datasets ===
train_dataset = Dataset.from_pandas(train_df.rename(columns={"target": "labels"}))
test_dataset = Dataset.from_pandas(test_df.rename(columns={"target": "labels"}))

# === Tokenizer & Tokenization ===
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_fn(example):
    return tokenizer(example["func"], truncation=True, padding="max_length", max_length=512)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# === Model (with safety for classification head) ===
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)
model.config.problem_type = "single_label_classification"


# === Custom metrics ===
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    except:
        tn = fp = fn = tp = 0

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "kappa": cohen_kappa_score(labels, preds),
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "auc": auc
    }

# === Training Arguments ===
training_args = TrainingArguments(
    output_dir="./codebert_devign_model_UP",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16 =True,
    report_to="none"
)

# === Trainer ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# === Train and Save ===
trainer.train()
trainer.save_model("./codebert_devign_model_UP")
tokenizer.save_pretrained("./codebert_devign_model_UP")



In [ ]:

import pandas as pd
nasa_df = pd.read_csv("nasa_data_vul.csv")
nasa_df.info()

In [ ]:
my_df = pd.read_csv("nasa_complete_func_deduplicated.csv")
my_df.info()

In [ ]:
# Assume the column name is 'my_column'
filtered_df2 = my_df[~my_df['old_func'].isin(nasa_df['Code'])]
filtered_df2.info()


In [ ]:
import pandas as pd

import re

import re

def is_meaningful_function(code, min_statements=2):
    if not isinstance(code, str):
        return False

    code = code.strip()

    if '{' not in code or '}' not in code:
        return False

    # Extract body
    try:
        open_idx = code.index('{')
        close_idx = code.rindex('}')
        body = code[open_idx + 1 : close_idx]
    except ValueError:
        return False

    # Remove block and inline comments
    body = re.sub(r'/\*.*?\*/', '', body, flags=re.DOTALL)
    body = re.sub(r'//.*', '', body)

    # Count statements (lines ending in semicolon)
    statement_count = body.count(';')

    return statement_count > min_statements


# === Load your DataFrame ===
# df = pd.read_csv("your_file.csv")  # already loaded as 'ddd' maybe?

# Apply filter
my_df = my_df[my_df["old_func"].apply(is_meaningful_function)].copy()

my_df.info()

In [ ]:
my_df.to_csv("nasa_second_part.csv",index=False,quoting=1)

In [ ]:
import pandas as pd
import subprocess
import os

# Change this to match your real CSV path
INPUT_CSV = "nasa_second_part.csv"
OUTPUT_CSV = "labeled_dataset.csv"

# Change this to match your real code column
CODE_COLUMN = "old_func"
LABEL_COLUMN = "label"

# Create a temp folder for code files
os.makedirs("tmp_code", exist_ok=True)

def run_cppcheck_on_snippet(code_string):
    """
    Runs cppcheck on a single code snippet.
    Returns 1 if any error or warning found, else 0.
    """
    tmp_file = "tmp_code/tmp_snippet.c"

    # Optionally wrap the code to make it valid C
    wrapped_code = f"""
#include <stdio.h>
#include <string.h>
void test_func() {{
{code_string}
}}
"""
    with open(tmp_file, "w") as f:
        f.write(wrapped_code)

    try:
        result = subprocess.run(
            ["cppcheck", "--enable=all", "--xml", tmp_file],
            stderr=subprocess.PIPE,
            stdout=subprocess.PIPE,
            encoding='utf-8'
        )
        cppcheck_output = result.stderr

        # cppcheck writes XML output to stderr
        if "<error" in cppcheck_output:
            return 1
        else:
            return 0
    except Exception as ex:
        print("Error running cppcheck:", ex)
        return 0

# === MAIN SCRIPT ===

# Load CSV
df = pd.read_csv(INPUT_CSV)

# Apply static analysis
print(f"Running cppcheck on {len(df)} code snippets...")
df[LABEL_COLUMN] = df[CODE_COLUMN].apply(run_cppcheck_on_snippet)

# Save new CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved labeled dataset to {OUTPUT_CSV}")


In [ ]:
df.info()

In [ ]:
df.to_csv(OUTPUT_CSV, index=False,quoting=1)

In [ ]:
import os
import xml.etree.ElementTree as ET
import subprocess

cppcheck_to_cwe = {
    "unusedFunction": "CWE-561",
    "variableScope": "CWE-768",
    "redundantInitialization": "CWE-563",
    "redundantAssignment": "CWE-563",
    "knownConditionTrueFalse": "CWE-571",
    "ConfigurationNotChecked": "CWE-696",
    "unreadVariable": "CWE-563",
    "pointerSize": "CWE-704",
    "invalidPrintfArgType_sint": "CWE-134",
    "invalidPrintfArgType_uint": "CWE-134",
    "arrayIndexOutOfBoundsCond": "CWE-129",
    "constParameter": "CWE-710",
    "unusedStructMember": "CWE-561",
    "funcArgNamesDifferent": "CWE-477"
}

def run_cppcheck_for_cwe(code_str):
    os.makedirs("tmp_code", exist_ok=True)
    tmp_file = "tmp_code/tmp.c"

    # Wrap the code snippet to be valid C
    wrapped_code = f"""
#include <stdio.h>
#include <string.h>
void test_func() {{
{code_str}
}}
"""

    with open(tmp_file, "w") as f:
        f.write(wrapped_code)

    try:
        result = subprocess.run(
            ["cppcheck", "--enable=all", "--xml", tmp_file],
            stderr=subprocess.PIPE,
            stdout=subprocess.PIPE,
            encoding='utf-8'
        )
        cppcheck_output = result.stderr

        # 🔎 Find the start of the actual XML
        xml_start = cppcheck_output.find("<?xml")
        if xml_start == -1:
            # No XML found at all (likely no errors)
            return "None"

        clean_xml = cppcheck_output[xml_start:]

        # Parse clean XML
        root = ET.fromstring(clean_xml)

        cwes = set()
        for error in root.iter("error"):
            error_id = error.get("id")
            if error_id in cppcheck_to_cwe:
                cwes.add(cppcheck_to_cwe[error_id])

        if cwes:
            return ";".join(sorted(cwes))
        else:
            return "None"

    except Exception as ex:
        print("Error parsing cppcheck output:", ex)
        return "None"


In [ ]:
df["cwe_id"] = df["old_func"].apply(run_cppcheck_for_cwe)
df.to_csv("nasa_second_part_with_CWE.csv",index=False,quoting=1)

In [ ]:
ds = pd.read_csv("nasa_data_vul.csv")
ds.info()

In [ ]:
ds['true_label'].value_counts()

In [ ]:
import pandas as pd

# Load your file


# Filter rows where true_label is 'vulnerable'
vulnerable_df = ds[ds['true_label'] == 'vulnerable']

# Show result
print(vulnerable_df.head())
print(f"Total vulnerable samples: {len(vulnerable_df)}")

# Save to new CSV
vulnerable_df.to_csv("nasa_dataset_first_part.csv", index=False)


In [ ]:
vulnerable_df.info()

In [ ]:
print(vulnerable_df['file'].iloc[0])

In [ ]:
print(df['repo'].iloc[0])

In [ ]:
import pandas as pd

# 1. Load the CSV
df = pd.read_csv("nasa_dataset_first_part.csv")

# 2. Define repo extraction
def extract_repo(path):
    marker = "nasa_all_repos/"
    if marker in path:
        rest = path.split(marker, 1)[1]
        return rest.split("/", 1)[0]
    else:
        return "UNKNOWN"

# 3. Apply to extract repo
df['repo'] = df['file'].apply(extract_repo)

# 4. Add func and label
df['func'] = df['Code']
df['label'] = 1

# 5. Keep only needed columns
new_df = df[['repo', 'func', 'label']]

# 6. Save
new_df.to_csv("nasa_part_1.csv", index=False)

print(" New dataset saved ")


In [ ]:
new_df.info()

In [ ]:
new_df.head()

In [ ]:
import pandas as pd

# Load the second dataset
second_df = pd.read_csv("nasa_second_part_with_CWE.csv")

# Take only the first 3000 rows
second_df = second_df.head(3000)

# Add the required columns
second_df['func'] = second_df['old_func']
second_df['label'] = 1

# Select only repo, func, label
new_second_df = second_df[['repo', 'func', 'label']]

# Save to CSV
new_second_df.to_csv("nasa_dataset_part_2.csv", index=False)

print("New second dataset saved ")


In [ ]:
new_second_df.info()

In [ ]:
combined_df = pd.concat([new_df, new_second_df], ignore_index=True)


In [ ]:
combined_df.to_csv('nasa_dataset_with_6k_vul.csv',index=False)
combined_df.info()

In [ ]:
df = pd.read_csv("nsvd_non_vulnerable_balanced.csv")
df.info()

In [ ]:
print(df['file'].iloc[1])

In [ ]:
other_df_6k = df.head(6000)
new_df = pd.DataFrame()
new_df['func'] = other_df_6k['Code']
new_df['label'] = 0
new_df.to_csv('nasa_non_vul.csv',index=False)

In [ ]:
nasa_df = pd.concat([combined_df,new_df], ignore_index=True)


In [ ]:
nasa_df.info()

In [ ]:
nasa_df.to_csv('nasa_complete-data_vul_non-vul.csv',index=False)

Testing nasa dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

nasa_train, nasa_test = train_test_split(
    nasa_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
nasa_train.to_csv('nasa_train.csv',index=False)
nasa_test.to_csv('nasa_test.csv',index=False)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)


In [ ]:


import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)


train_df = pd.read_csv("nasa_train.csv")
test_df = pd.read_csv("nasa_test.csv")

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

#
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)


model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["func"], truncation=True)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    except:
        tn = fp = fn = tp = 0

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "kappa": cohen_kappa_score(labels, preds),
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "auc": auc
    }


training_args = TrainingArguments(
    output_dir="./codebert-nasa-finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available()  # enable mixed precision if GPU
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


trainer.train()


results = trainer.evaluate()
print(results)


In [ ]:
trainer.save_model("./codebert-nasa-finetuned")

devign_test

In [ ]:
devign_test = pd.read_csv('devign_test.csv')
devign_test.info()

In [ ]:
import pandas as pd

# Load your CSV
df = pd.read_csv("devign_test.csv")

# Rename columns
df = df.rename(columns={"input": "func", "output": "label"})

# Keep only the needed columns
df = df[["func", "label"]]




In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"



import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    except:
        tn = fp = fn = tp = 0

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "kappa": cohen_kappa_score(labels, preds),
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "auc": auc
    }

# ✅ LOAD YOUR FINETUNED MODEL
model_path = "./codebert-nasa-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ✅ TESTING FUNCTION
def test_on_csv(df):


    # Load the CSV

    dataset = Dataset.from_pandas(df)

    # Tokenize
    def tokenize_fn(batch):
        return tokenizer(batch["func"], truncation=True)
    dataset = dataset.map(tokenize_fn, batched=True)

    # Create Trainer for evaluation
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Evaluate
    results = trainer.evaluate(eval_dataset=dataset)
    print("\n Evaluation Results:")
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    return results

test_on_csv(df)


In [ ]:
import os
print(os.listdir("./codebert-nasa-finetuned"))


In [ ]:
df = pd.read_csv("diverse_test.csv")
df.info()

In [ ]:
tokenizer.save_pretrained("./codebert-nasa-finetuned")

In [ ]:

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    except:
        tn = fp = fn = tp = 0

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "kappa": cohen_kappa_score(labels, preds),
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "auc": auc
    }


model_path = "./graphcodebert_devign_model_UP"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def test_on_any_csv(csv_file, input_col, output_col):
    print(f"\n Evaluating on: {csv_file}")

    # Load CSV
    df = pd.read_csv(csv_file)


    df = df.rename(columns={input_col: "func", output_col: "label"})


    df = df.dropna(subset=["func", "label"]).copy()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])]

    print(f"Loaded {len(df)} samples with labels:", df["label"].unique())
    print(df["label"].value_counts())

    if len(df) == 0:
        print(" No valid samples in this test set after cleaning. Exiting.")
        return


    dataset = Dataset.from_pandas(df)


    def tokenize_fn(batch):
        return tokenizer(batch["func"], truncation=True)
    dataset = dataset.map(tokenize_fn, batched=True)


    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    results = trainer.evaluate(eval_dataset=dataset)
    print("\n Evaluation Results:")
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    return results


# For your Diverse test CSV with columns: code_snip, output
test_on_any_csv("diverse_test.csv", input_col="code_snip", output_col="output")


In [ ]:
def clean_and_validate(df):
    print("\n🔎 Before cleaning:")
    print(df["label"].value_counts(dropna=False))

    # Drop NaN
    df = df.dropna(subset=["func", "label"]).copy()

    # Convert to numeric (coerce errors to NaN)
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    # Drop NaN again
    df = df.dropna(subset=["label"])

    # Convert to int
    df["label"] = df["label"].astype(int)

    # Filter only 0 and 1
    df = df[df["label"].isin([0, 1])]

    print("\n After cleaning:")
    print(df["label"].value_counts())
    print(f"Remaining samples: {len(df)}")

    return df


In [ ]:

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    except:
        tn = fp = fn = tp = 0

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "kappa": cohen_kappa_score(labels, preds),
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "auc": auc
    }


model_path = "./graphcodebert_devign_model_UP"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def clean_dataframe(df):
    print("\n Before cleaning:")
    print(df["label"].value_counts(dropna=False))

    # Drop empty rows
    df = df.dropna(subset=["func", "label"]).copy()

    # Force numeric
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)

    # Filter only 0 or 1
    df = df[df["label"].isin([0, 1])]

    print("\n After cleaning:")
    print(df["label"].value_counts())
    print(f"Remaining samples: {len(df)}")

    return df


def test_on_any_csv(csv_file, input_col, output_col):
    print(f"\n🔎 Evaluating on: {csv_file}")

    # Load CSV
    df = pd.read_csv(csv_file)
    print(f"CSV loaded with {len(df)} rows.")

    df = df.rename(columns={input_col: "func", output_col: "label"})

    df = clean_dataframe(df)

    if len(df) == 0:
        print(" No valid samples left after cleaning. Exiting.")
        return

    #
    dataset = Dataset.from_pandas(df)


    def tokenize_fn(batch):
        return tokenizer(batch["func"], truncation=True)
    dataset = dataset.map(tokenize_fn, batched=True)

    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )


    results = trainer.evaluate(eval_dataset=dataset)
    print("\n Evaluation Results:")
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    return results


# For your Diverse test CSV with columns: code_snip, output
test_on_any_csv("diverse_test.csv", input_col="code_snip", output_col="output")


In [ ]:
import numpy as np
import torch
import pandas as pd
import time
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)

# ====== STEP 1: Load Data ======
train_df = pd.read_csv('nasa_train.csv')
test_df = pd.read_csv('nasa_test.csv')



# Drop rows with missing values
train_df = train_df.dropna(subset=["func", "label"])
test_df = test_df.dropna(subset=["func", "label"])

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# Convert to HuggingFace Datasets
train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

# ====== STEP 2: Tokenizer Setup ======
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        ["FUNC: " + c for c in batch["func"]],
        truncation=True, padding='max_length', max_length=256
    )

# Apply tokenizer
train_tok = train_ds.map(tokenize_fn, batched=True)
test_tok  = test_ds.map(tokenize_fn, batched=True)

# ====== STEP 3: Prepare datasets ======
train_tok = train_tok.rename_column('label', 'labels')
test_tok  = test_tok.rename_column('label', 'labels')
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# ====== STEP 4: Load Model ======
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.problem_type = "single_label_classification"
model.config.id2label = {0: "safe", 1: "vuln"}
model.config.label2id = {"safe": 0, "vuln": 1}

# ====== STEP 5: Metrics ======
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr":         fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1":          f1_score(labels, preds, zero_division=0),
        "mcc":         matthews_corrcoef(labels, preds),
        "kappa":       cohen_kappa_score(labels, preds),
        "mse":         mean_squared_error(labels, preds),
        "mae":         mean_absolute_error(labels, preds),
        "auc":         auc
    }

# ====== STEP 6: Training Setup ======
data_collator = DataCollatorWithPadding(tokenizer)
args = TrainingArguments(
    output_dir='.//codebert_nasa_finetuned_model_UP',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ====== STEP 7: Train ======
start_train = time.time()
trainer.train()
end_train = time.time()
train_runtime = end_train - start_train
print(f"\n=== Training completed in {train_runtime:.2f} seconds ({train_runtime / 60:.2f} minutes) ===")

# ====== STEP 8: Evaluate ======
start_eval = time.time()
metrics = trainer.evaluate()
end_eval = time.time()
eval_runtime = end_eval - start_eval
print(f"\n=== Evaluation completed in {eval_runtime:.2f} seconds ({eval_runtime / 60:.2f} minutes) ===")

print("\n=== Hold-out Metrics (Trained and Tested with FUNC only) ===")
for key, value in metrics.items():
    if key.startswith("eval_"):
        print(f"{key[5:]:<12}: {value:.4f}")

print(f"Train Time (s): {train_runtime:.2f}")
print(f"Eval Time (s):  {eval_runtime:.2f}")

# ====== STEP 9: Save Model ======
save_dir = './codebert_nasa_finetuned_model_UP'
trainer.save_model(save_dir)
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\nModel and tokenizer saved to {save_dir}")


In [ ]:
# ====== STEP 0: Imports ======
import numpy as np
import torch
import pandas as pd
import time
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)

# ====== STEP 1: Load and Clean Data ======
print("\n=== Loading Data ===")
train_df = pd.read_csv('nasa_train.csv')
test_df = pd.read_csv('nasa_test.csv')

# Drop rows with missing 'func'
train_df = train_df.dropna(subset=["func"])
test_df = test_df.dropna(subset=["func"])

# Clean label column
# 1. Coerce non-numeric labels to NaN
train_df["label"] = pd.to_numeric(train_df["label"], errors="coerce")
test_df["label"] = pd.to_numeric(test_df["label"], errors="coerce")

# 2. Drop rows with NaN labels
train_df = train_df.dropna(subset=["label"])
test_df = test_df.dropna(subset=["label"])

# 3. Convert to int
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# 4. Filter for valid labels (0 or 1 only)
train_df = train_df[train_df["label"].isin([0, 1])]
test_df = test_df[test_df["label"].isin([0, 1])]

print("\n=== Label Counts (Train) ===")
print(train_df["label"].value_counts())
print("\n=== Label Counts (Test) ===")
print(test_df["label"].value_counts())

# ====== STEP 2: Convert to HF Datasets ======
train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

# ====== STEP 3: Tokenizer Setup ======
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        ["FUNC: " + c for c in batch["func"]],
        truncation=True,
        padding='max_length',
        max_length=256
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
test_tok  = test_ds.map(tokenize_fn, batched=True)

train_tok = train_tok.rename_column('label', 'labels')
test_tok  = test_tok.rename_column('label', 'labels')
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# ====== STEP 4: Load Model ======
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.problem_type = "single_label_classification"
model.config.id2label = {0: "safe", 1: "vuln"}
model.config.label2id = {"safe": 0, "vuln": 1}

# ====== STEP 5: Metrics ======
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr":         fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1":          f1_score(labels, preds, zero_division=0),
        "mcc":         matthews_corrcoef(labels, preds),
        "kappa":       cohen_kappa_score(labels, preds),
        "mse":         mean_squared_error(labels, preds),
        "mae":         mean_absolute_error(labels, preds),
        "auc":         auc
    }

# ====== STEP 6: Training Setup ======
data_collator = DataCollatorWithPadding(tokenizer)
args = TrainingArguments(
    output_dir='./codebert_nasa_finetuned_model_UP',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ====== STEP 7: Train ======
start_train = time.time()
trainer.train()
end_train = time.time()
train_runtime = end_train - start_train
print(f"\n=== Training completed in {train_runtime:.2f} seconds ({train_runtime / 60:.2f} minutes) ===")

# ====== STEP 8: Evaluate ======
start_eval = time.time()
metrics = trainer.evaluate()
end_eval = time.time()
eval_runtime = end_eval - start_eval
print(f"\n=== Evaluation completed in {eval_runtime:.2f} seconds ({eval_runtime / 60:.2f} minutes) ===")

print("\n=== Hold-out Metrics (FUNC only) ===")
for key, value in metrics.items():
    if key.startswith("eval_"):
        print(f"{key[5:]:<12}: {value:.4f}")

print(f"Train Time (s): {train_runtime:.2f}")
print(f"Eval Time (s):  {eval_runtime:.2f}")

# ====== STEP 9: Save Model ======
save_dir = './codebert_nasa_finetuned_model_UP'
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\nModel and tokenizer saved to {save_dir}")


In [ ]:
# ====== STEP 0: Imports ======
import numpy as np
import torch
import pandas as pd
import time
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score
)

# ====== STEP 1: Load and Clean Data ======
print("\n=== Loading Data ===")
train_df = pd.read_csv('nasa_train.csv')
test_df = pd.read_csv('nasa_test.csv')

# Drop rows with missing 'func'
train_df = train_df.dropna(subset=["func"])
test_df = test_df.dropna(subset=["func"])

# Convert label column to numeric, drop NaNs
train_df["label"] = pd.to_numeric(train_df["label"], errors="coerce")
test_df["label"] = pd.to_numeric(test_df["label"], errors="coerce")
train_df = train_df.dropna(subset=["label"])
test_df = test_df.dropna(subset=["label"])

# Convert to int
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# Filter for valid labels (0 or 1)
train_df = train_df[train_df["label"].isin([0, 1])]
test_df = test_df[test_df["label"].isin([0, 1])]

print("\n=== Label Counts (Train) ===")
print(train_df["label"].value_counts())
print("\n=== Label Counts (Test) ===")
print(test_df["label"].value_counts())

# ====== STEP 2: Convert to HF Datasets ======
train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

# ====== STEP 3: Tokenizer Setup ======
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        ["FUNC: " + c for c in batch["func"]],
        truncation=True,
        padding='max_length',
        max_length=256
    )

# Ensure 'label' column is preserved
keep_cols = ['func', 'label']

train_tok = train_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[col for col in train_ds.column_names if col not in keep_cols]
)

test_tok = test_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[col for col in test_ds.column_names if col not in keep_cols]
)

# Rename label column for Trainer
train_tok = train_tok.rename_column('label', 'labels')
test_tok  = test_tok.rename_column('label', 'labels')

# Set format for PyTorch
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# ====== STEP 4: Sanity Check ======
print("\n=== Example tokenized Train Sample ===")
print(train_tok[0])

assert all(l in [0, 1] for l in train_tok['labels']), "Error: invalid label in train_tok!"
assert all(l in [0, 1] for l in test_tok['labels']), "Error: invalid label in test_tok!"

# ====== STEP 5: Load Model ======
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.problem_type = "single_label_classification"
model.config.id2label = {0: "safe", 1: "vuln"}
model.config.label2id = {"safe": 0, "vuln": 1}

# ====== STEP 6: Metrics ======
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "fpr":         fp / (fp + tn) if (fp + tn) > 0 else 0,
        "f1":          f1_score(labels, preds, zero_division=0),
        "mcc":         matthews_corrcoef(labels, preds),
        "kappa":       cohen_kappa_score(labels, preds),
        "mse":         mean_squared_error(labels, preds),
        "mae":         mean_absolute_error(labels, preds),
        "auc":         auc
    }

# ====== STEP 7: Training Setup ======
data_collator = DataCollatorWithPadding(tokenizer)
args = TrainingArguments(
    output_dir='./codebert_nasa_finetuned_model_UP',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ====== STEP 8: Train ======
start_train = time.time()
trainer.train()
end_train = time.time()
train_runtime = end_train - start_train
print(f"\n=== Training completed in {train_runtime:.2f} seconds ({train_runtime / 60:.2f} minutes) ===")

# ====== STEP 9: Evaluate ======
start_eval = time.time()
metrics = trainer.evaluate()
end_eval = time.time()
eval_runtime = end_eval - start_eval
print(f"\n=== Evaluation completed in {eval_runtime:.2f} seconds ({eval_runtime / 60:.2f} minutes) ===")

print("\n=== Hold-out Metrics (FUNC only) ===")
for key, value in metrics.items():
    if key.startswith("eval_"):
        print(f"{key[5:]:<12}: {value:.4f}")

print(f"Train Time (s): {train_runtime:.2f}")
print(f"Eval Time (s):  {eval_runtime:.2f}")

# ====== STEP 10: Save Model ======
save_dir = './codebert_nasa_finetuned_model_UP'
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\nModel and tokenizer saved to {save_dir}")


In [ ]:
import pandas as pd

nas= pd.read_csv('nasa_complete-data_vul_non-vul.csv')
nas.info()

In [ ]:
repo_counts = nas['repo'].value_counts(dropna=False)

print(repo_counts)

In [ ]:
cwe_counts = nas['label'].value_counts()
print(cwe_counts)


In [ ]:
total_rows = len(nas)
rows_with_repo = nas['repo'].notna().sum()

print("Total rows:", total_rows)
print("Rows with repo:", rows_with_repo)
print("Rows without repo:", total_rows - rows_with_repo)


In [ ]:
import matplotlib.pyplot as plt

# Count samples per repo (including NaN if needed)
repo_counts = nas['repo'].value_counts()

# Plot the top N (for example top 10)
top_n = 10
repo_counts.head(top_n).plot(kind='bar', figsize=(10,6))

plt.title('Distribution of Function Samples Across Top 10 NASA Repositories')
plt.xlabel('Repository')
plt.ylabel('Number of Samples')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Count occurrences
repo_counts = nas['repo'].value_counts()

# Plot top 10 repositories
top_n = 20
repo_counts.head(top_n).plot(kind='bar', figsize=(10,6), color='steelblue')

plt.title('Distribution of Function Samples per NASA Repository', fontsize=14)
plt.xlabel('Repository', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()
